# Korteweg-de Vries equation

$$u_t + \eta\,u\,u_x + \mu^2\,u_{xxx} = 0,\qquad x\in(-1,1),\ t\in(0,1),\qquad u(x,0)=\cos(\pi x)$$

A dispersive equation with periodic boundaries.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

import pinn
from pinn import operators as op, sampling

%matplotlib inline
jax.config.update("jax_enable_x64", True)   # double precision

In [ ]:
X, T = 0, 1


class KdVProblem(pinn.Problem):
    """u_t + eta u u_x + mu^2 u_xxx = 0, periodic on (-1, 1)."""

    x_min, x_max, t_max = -1.0, 1.0, 1.0
    eta, mu = 1.0, 0.022
    problem_name = "KdV"
    ref_path = pinn.reference_path("kdv")

    def __init__(self, *, n_pde, n_ic, n_bc=0):
        self.n_pde, self.n_ic, self.n_bc = n_pde, n_ic, n_bc

    def residual_fns(self):
        return {"pde": self.pde_residual, "ic": self.ic_residual}

    def pde_residual(self, model, coords):
        u = lambda c: model(c)[0]
        r = (op.grad(u, coords, T)
             + self.eta * u(coords) * op.grad(u, coords, X)
             + self.mu**2 * op.grad_n(u, coords, X, 3))
        return jnp.array([r])

    def ic_residual(self, model, coords):
        return jnp.array([model(coords[:2])[0] - coords[2]])

    def samplers(self):
        return {"pde": sampling.interior(self.x_min, self.x_max),
                "ic":  sampling.initial_line(self.x_min, self.x_max)}

    def analytical_ic(self, x):
        return jnp.cos(jnp.pi * x)

In [ ]:
cfg = pinn.RunConfig(
    dt=0.25,
    network=lambda key: pinn.SIREN(
        key, KdVProblem, hidden_dims=(60, 60, 60, 60),
        periodic_bc=True, n_inputs=2, n_outputs=1,
    ),
    n_pde=2**15, n_ic=2**14,
    residual_sketch=4000, parameter_sketch=4000,
    batch_size=2**14, probe_batch_size=2**10,
    pde_weight=1e-4,
)

In [ ]:
pinn.precompile64(KdVProblem, cfg)

In [ ]:
results = pinn.train64(KdVProblem, cfg)

## Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref = pinn.load_reference("kdv")
x0, x1 = results["plot_x0"], results["plot_x1"]
lx, ly = results["plot_axes"]
extent = [x0[0], x0[-1], x1[0], x1[-1]]

for ch in ref.channels:
    pred = np.array(results["u_pred_plot"][ch])
    exact = np.array(ref.plot_grids[ch])
    err = np.abs(pred - exact)
    rel = np.linalg.norm(pred - exact) / np.linalg.norm(exact)

    fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, data, title, cmap in zip(
        axs, [pred, exact, err],
        [f"PINN  ${ch}$", f"reference  ${ch}$", f"abs error  (rel $\\ell_2$={rel:.2e})"],
        ["RdBu_r", "RdBu_r", "magma"],
    ):
        im = ax.imshow(data.T, origin="lower", aspect="auto", extent=extent, cmap=cmap)
        ax.set(xlabel=f"${lx}$", ylabel=f"${ly}$", title=title)
        fig.colorbar(im, ax=ax)
    plt.show()